In [1]:
import sys, json
sys.path.append("../backend")
import pandas as pd, joblib
from features import extract_features
from sklearn.metrics import classification_report, confusion_matrix

model = joblib.load("../backend/model.pkl")
cols = joblib.load("../backend/feature_columns.pkl")
try:
    threshold = json.load(open("../backend/threshold.json"))["threshold"]
except FileNotFoundError:
    threshold = 0.7
print("Threshold:", threshold)

with open("../data/openphish.txt") as f:
    phish = [l.strip() for l in f if l.strip()][:500]

tr = pd.read_csv("../data/top-1m.csv", header=None, names=["rank", "domain"]).head(500)
legit = ["https://" + d for d in tr["domain"]]

urls = phish + legit
y = [1] * len(phish) + [0] * len(legit)
print(len(phish), "phishing,", len(legit), "legitimate")

Threshold: 0.7
300 phishing, 500 legitimate


In [2]:
X = pd.DataFrame([extract_features(u) for u in urls])[cols]
proba = model.predict_proba(X)[:, 1]
pred = (proba >= threshold).astype(int)

print(classification_report(y, pred, target_names=["Legitimate", "Phishing"]))
print(confusion_matrix(y, pred))

res = pd.DataFrame({"url": urls, "label": y, "score": proba})

              precision    recall  f1-score   support

  Legitimate       0.69      0.99      0.82       500
    Phishing       0.93      0.28      0.43       300

    accuracy                           0.72       800
   macro avg       0.81      0.63      0.62       800
weighted avg       0.78      0.72      0.67       800

[[494   6]
 [217  83]]


In [5]:
pd.set_option("display.max_colwidth", 90)
print("FALSE ALARMS (legitimate flagged as phishing):")
print(res[(res.label == 0) & (res.score >= threshold)].sort_values("score", ascending=False).head(15))

print("\nMISSED PHISHING (not flagged):")
print(res[(res.label == 1) & (res.score < threshold)].sort_values("score").head(15))

FALSE ALARMS (legitimate flagged as phishing):
                         url  label     score
622  https://skybridge.click      0  0.980000
424        https://cdn77.org      0  0.727634
613  https://byteoversea.net      0  0.720208
591  https://dnsmadeeasy.com      0  0.720208
387           https://f5.com      0  0.701270
488           https://w3.org      0  0.701270

MISSED PHISHING (not flagged):
                                                                   url  label  \
257        https://mainease.com/BP9SGCTK0422-the-evolution-of-empathy/      1   
240             https://1url.at/www/roblox-users-4802576232415-profile      1   
70   http://wunderschone.sbs/form/f6c8189e-aa57-42c4-a028-6d40ad2cdf76      1   
46                              https://facebookfakepage.blogspot.com/      1   
82                       https://online-facebook-privacy.blogspot.com/      1   
171        https://tmbwqter.com/d/b45ca627-530a-46a2-88f8-7756bd0bd9a8      1   
93                     https://r

In [6]:
tr = pd.read_csv("../data/top-1m.csv", header=None, names=["rank", "domain"]).head(10000)
tr["domain"].to_csv("../backend/top_domains.txt", index=False, header=False)

In [8]:
tr = pd.read_csv("../data/top-1m.csv", header=None, names=["rank", "domain"]).head(10000)
tr["domain"].to_csv("../backend/top_domains.txt", index=False, header=False)

In [3]:
import numpy as np
y_arr = np.array(y)

for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    p = (proba >= t).astype(int)
    caught = int(((p == 1) & (y_arr == 1)).sum())
    false_alarms = int(((p == 1) & (y_arr == 0)).sum())
    print(f"threshold {t}: caught {caught}/{len(phish)} phishing | "
          f"false alarms {false_alarms}/{len(legit)} legit")

threshold 0.1: caught 297/300 phishing | false alarms 500/500 legit
threshold 0.2: caught 294/300 phishing | false alarms 500/500 legit
threshold 0.3: caught 264/300 phishing | false alarms 500/500 legit
threshold 0.4: caught 232/300 phishing | false alarms 500/500 legit
threshold 0.5: caught 189/300 phishing | false alarms 408/500 legit
threshold 0.6: caught 148/300 phishing | false alarms 170/500 legit
threshold 0.7: caught 83/300 phishing | false alarms 6/500 legit
